In [28]:
# Imports
import pandas as pd
import pydeck
from geolib import geohash as geolib

In [29]:
# Geohashes in switzerland
swiss_geohashes = [f"u0{c}" for c in "kmqrhjnp"]

In [30]:
def merge_consecutive_movements(df, time_min, geohash_precision=5):
    df = df.sort_values(['participant_id', 'mean_of_transport', 'start_time'])

    df['prev_end_time'] = df.groupby(['participant_id', 'mean_of_transport'])['end_time'].shift()
    df['prev_end_geohash'] = df.groupby(['participant_id', 'mean_of_transport'])['end_geohash'].shift()
    df['prev_start_geohash'] = df.groupby(['participant_id', 'mean_of_transport'])['start_geohash'].shift()
    
    df['time_diff'] = (df['start_time'] - pd.to_datetime(df['prev_end_time'])).dt.total_seconds() / 60

    # Truncate geohashes to compare at a higher level (less precise = broader area)
    df['start_geo_upper'] = df['start_geohash'].str[:geohash_precision]
    df['end_geo_upper'] = df['end_geohash'].str[:geohash_precision]
    df['prev_start_geo_upper'] = df['prev_start_geohash'].str[:geohash_precision]
    df['prev_end_geo_upper'] = df['prev_end_geohash'].str[:geohash_precision]
    
    # Check if current trip connects to previous trip (prev end → current start)
    # AND if they're part of the same route (same general start/end areas)
    df['should_merge'] = (
        (df['time_diff'] <= time_min) &  # Time is close enough
        (df['start_geo_upper'] == df['prev_start_geo_upper']) &  # Same general start area
        (df['end_geo_upper'] == df['prev_end_geo_upper'])  # Same general end area
    )

    # Create groups based on breaks in the merge condition
    df['group'] = (~df['should_merge']).cumsum()

    df = df.groupby(['participant_id', 'mean_of_transport', 'group']).agg(
        start_time=('start_time', 'first'),
        end_time=('end_time', 'last'),
        start_geohash=('start_geohash', 'first'),
        end_geohash=('end_geohash', 'last'),
        distance=('distance(m)', 'sum'),
        gCO2=('gCO2', 'sum'),
        mean_of_transport=('mean_of_transport', 'first')
    ).reset_index(drop=True)

    df.rename(columns={'distance': 'distance(m)'}, inplace=True)

    return df

In [31]:
# Read geohashes to coordinates mapping (dataframe with columns: geohash, latitude, longitude)
geo_to_coords = pd.read_pickle('geohashes_to_coords.pkl')

# Function to complete geo_to_coords with missing geohashes
def complete_geo_to_coords(geo_to_coords, geohashes):
    missing_geohashes = set(geohashes) - set(geo_to_coords['geohash'])
    new_rows = []
    for gh in missing_geohashes:
        lat, lon = geolib.decode(gh)
        new_rows.append({'geohash': gh, 'latitude': lat, 'longitude': lon})
    if new_rows:
        geo_to_coords = pd.concat([geo_to_coords, pd.DataFrame(new_rows)], ignore_index=True)
        geo_to_coords.to_pickle('geohashes_to_coords.pkl') # Save to file
    return geo_to_coords

In [32]:
from sklearn.cluster import DBSCAN
import numpy as np
import pandas as pd

def snap_close_geohashes_fast(df_t, mode='Train', max_distance_km=2):
    df_mode = df_t[df_t['mean_of_transport'] == mode].copy()
    if df_mode.empty:
        return df_t

    # Keep only valid coordinates
    df_mode = df_mode[df_mode['start_coords'].notna() & df_mode['end_coords'].notna()].copy()
    
    if df_mode.empty:
        return df_t

    # Combine start AND end coordinates for clustering
    start_coords = np.array([[lat, lon] for lon, lat in df_mode['start_coords']])
    end_coords = np.array([[lat, lon] for lon, lat in df_mode['end_coords']])
    
    # Stack them horizontally: [start_lat, start_lon, end_lat, end_lon]
    combined_coords = np.hstack([start_coords, end_coords])
    
    kms_per_radian = 6371.0088
    epsilon = max_distance_km / kms_per_radian

    # Custom distance function: max of start distance and end distance
    def trip_distance(trip1, trip2):
        start1, end1 = trip1[:2], trip1[2:]
        start2, end2 = trip2[:2], trip2[2:]
        
        # Haversine distance for start and end
        start_dist = haversine_distance(start1, start2)
        end_dist = haversine_distance(end1, end2)
        
        # Both start and end must be close
        return max(start_dist, end_dist)
    
    def haversine_distance(coord1, coord2):
        lat1, lon1 = np.radians(coord1)
        lat2, lon2 = np.radians(coord2)
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        return 2 * np.arcsin(np.sqrt(a))

    # Cluster by both start and end coordinates
    db = DBSCAN(eps=epsilon, min_samples=1, metric=trip_distance).fit(combined_coords)
    df_mode['cluster'] = db.labels_

    # Weighted majority geohash per cluster
    merged_clusters = []
    for c in df_mode['cluster'].unique():
        cluster_rows = df_mode[df_mode['cluster'] == c]
        
        starts = cluster_rows.groupby('start_geohash')['count'].sum()
        ends = cluster_rows.groupby('end_geohash')['count'].sum()
        most_common_start = starts.idxmax()
        most_common_end = ends.idxmax()

        total_count = cluster_rows['count'].sum()

        merged_clusters.append(pd.DataFrame({
            'start_geohash': [most_common_start],
            'end_geohash': [most_common_end],
            'mean_of_transport': [mode],
            'count': [total_count]
        }))

    # Combine merged clusters
    df_mode_new = pd.concat(merged_clusters, ignore_index=True)

    # Compute the new start and end coordinates
    df_mode_new = df_mode_new.merge(
        geo_to_coords.rename(columns={'geohash': 'start_geohash', 'latitude': 'start_latitude', 'longitude': 'start_longitude'}),
        on='start_geohash', how='left'
    )
    df_mode_new = df_mode_new.merge(
        geo_to_coords.rename(columns={'geohash': 'end_geohash', 'latitude': 'end_latitude', 'longitude': 'end_longitude'}),
        on='end_geohash', how='left'
    )
    df_mode_new['start_coords'] = list(zip(df_mode_new['start_longitude'], df_mode_new['start_latitude']))
    df_mode_new['end_coords'] = list(zip(df_mode_new['end_longitude'], df_mode_new['end_latitude']))

    # Convert to list of floats
    df_mode_new['start_coords'] = df_mode_new['start_coords'].apply(lambda x: [float(coord) for coord in x])
    df_mode_new['end_coords'] = df_mode_new['end_coords'].apply(lambda x: [float(coord) for coord in x])

    df_mode_new = df_mode_new.drop(columns=['start_latitude', 'start_longitude', 'end_latitude', 'end_longitude'])

    # Replace only the clustered subset in the original DataFrame
    df_t = pd.concat([df_t[df_t['mean_of_transport'] != mode], df_mode_new], ignore_index=True)

    return df_t

In [33]:
# Read the data from the csv
def load_data(path):
    # Read the data from the csv
    df = pd.read_csv(path, sep=";")

    # Remove timezone
    df['start_time'] = df['start_time'].str[:-6]
    df['end_time'] = df['end_time'].str[:-6]

    # Transform the start_ and end_date to datetimes
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time'] = pd.to_datetime(df['end_time'])

    ## Convert distance to int
    df['distance(m)'] = df['distance(m)'].astype(int)

    ## Convert gCO2 to int
    df['gCO2'] = df['gCO2'].astype(int)

    # IF two movements from the same user have the same start geohash and start time, we should merge them into a single movement (keep the one with the highest distance)
    df['start_time'] = pd.to_datetime(df['start_time'])

    # Remove walks longer than 5km
    df = df[~((df['mean_of_transport'] == 'WALKING') & (df['distance(m)'] > 5000))]

    # Group by participant_id, start_geohash, start_time and maximum distance, keep all the other data from the row with the maxium distance
    df = df.groupby(['participant_id', 'start_geohash', 'start_time'])['distance(m)'].max().reset_index().merge(df, on=['participant_id', 'start_geohash', 'start_time', 'distance(m)'])

    return df

In [34]:
df = load_data('data/all_movements_112.csv')

In [35]:
# READ heatmap data
df_hm = pd.read_csv('data/all_paths_110.csv', sep=';')

In [36]:
# Get distribution of mean_of_transport = WALKING
df_walk = df[df['mean_of_transport'] == 'WALKING']
walk_distribution = df_walk['distance(m)'].describe()
print("WALKING distance distribution (m):")
print(walk_distribution)

WALKING distance distribution (m):
count    13039.000000
mean       796.526804
std        692.670074
min        250.000000
25%        369.000000
50%        571.000000
75%        919.500000
max       4984.000000
Name: distance(m), dtype: float64


In [37]:
# Keep data only in Switzerland
print(f"Initial data size: {len(df)}")
df = df[df['start_geohash'].str[:3].isin(swiss_geohashes)]
df = df[df['end_geohash'].str[:3].isin(swiss_geohashes)]
print(f"Data size after filtering to Switzerland: {len(df)}")

# Same for heatmap data
print(f"Initial heatmap data size: {len(df_hm)}")
df_hm = df_hm[df_hm['geohash'].str[:3].isin(swiss_geohashes)]
print(f"Heatmap data size after filtering to Switzerland: {len(df_hm)}")

Initial data size: 33825
Data size after filtering to Switzerland: 30615
Initial heatmap data size: 1277381
Heatmap data size after filtering to Switzerland: 1106731


In [38]:
# Remove data that have different mean_of_transport and original_mean_of_transport
df = df[df['mean_of_transport'] == df['original_mean_of_transport']]

In [39]:
# Merge consecutive movements with the same mode of transport within 20 minutes
df = merge_consecutive_movements(df, time_min=30, geohash_precision=4)

In [40]:
# Remove rows that have the same start and end geohashes
df = df[df['start_geohash'] != df['end_geohash']]

In [41]:
# Transform mode of transport
mots = {
    "CAR": "Car",
    "ELECTRIC_CAR": "Car",
    "HYBRID_CAR": "Car",
    "TRAIN": "Train",
    "WALKING": "Walking",
    "ON_BICYCLE": "Bicycle",
    "ELECTRIC_BIKE": "Bicycle",
    "SCOOTER": "Bicycle",
    "BUS": "Bus",
    "ELECTRIC_BUS": "Bus",
    "COACH": "Bus",
    "TRAM": "Tram",
    "PLANE": "Plane",
    "BOAT": "Boat",
    "BOAT_NO_ENGINE": "Boat"
}

df['mean_of_transport'] = df['mean_of_transport'].map(mots)

In [42]:
df

,start_time,end_time,start_geohash,end_geohash,distance(m),gCO2,mean_of_transport
0,2025-01-28 17:03:28,2025-01-28 17:34:50,u0qj6w,u0qj5x,8568,1130,Bus
1,2025-01-28 16:07:52,2025-01-28 16:53:06,u0mgth,u0qjd2,49134,343,Train
2,2025-01-28 16:57:29,2025-01-28 17:01:17,u0qjd2,u0qj6w,1581,0,Walking
3,2025-01-28 17:34:51,2025-01-28 17:48:06,u0qj5x,u0qj78,471,0,Walking
4,2025-05-23 15:38:13,2025-05-23 16:00:37,u0qd1e,u0q98x,10117,1335,Bus
...,...,...,...,...,...,...,...
26800,2025-10-02 14:08:42,2025-10-02 14:10:58,u0rcrh,u0rcrm,1546,0,Walking
26803,2025-10-11 04:27:03,2025-10-11 04:40:14,u0qnnr,u0qnnq,263,0,Walking
26807,2025-10-11 19:28:55,2025-10-11 19:37:48,u0qjd3,u0qj9c,1022,0,Walking
26808,2025-10-11 20:30:20,2025-10-11 20:41:10,u0qnnw,u0qnnq,701,0,Walking


In [43]:
# Group movements by start and end geohash and mean_of_transport and get the count of movements. Do not consider direction, so start_geohash A to end_geohash B is the same as start_geohash B to end_geohash A
df['geohash_pair'] = df.apply(lambda row: tuple(sorted([row['start_geohash'], row['end_geohash']])), axis=1)
df[['start_geohash', 'end_geohash']] = pd.DataFrame(df['geohash_pair'].tolist(), index=df.index)
df = df.drop(columns=['geohash_pair'])
grouped = df.groupby(['start_geohash', 'end_geohash', 'mean_of_transport']).size().reset_index(name='count')
grouped

,start_geohash,end_geohash,mean_of_transport,count
0,u0h43v,u0hhnz,Bicycle,1
1,u0hhg7,u0hqg7,Car,1
2,u0hhg7,u0hqge,Car,1
3,u0hhzk,u0hjqg,Car,1
4,u0hjzs,u0hnnk,Bicycle,1
...,...,...,...,...
7809,u0rcsv,u0rcw0,Car,1
7810,u0rctf,u0rctg,Walking,1
7811,u0rhjk,u0rk5n,Train,1
7812,u0rn1e,u0rn1u,Walking,1


In [44]:
# Convert geohashes to coordinates
all_geohashes = set(grouped['start_geohash']).union(set(grouped['end_geohash']))
geo_to_coords = complete_geo_to_coords(geo_to_coords, all_geohashes)

# Translate all start and end geohashes to coordinates
start_coords = grouped['start_geohash'].map(geo_to_coords.set_index('geohash')[['latitude', 'longitude']].to_dict(orient='index'))
end_coords = grouped['end_geohash'].map(geo_to_coords.set_index('geohash')[['latitude', 'longitude']].to_dict(orient='index'))

grouped['start_coords'] = start_coords.apply(lambda x: [float(x['longitude']), float(x['latitude'])])
grouped['end_coords'] = end_coords.apply(lambda x: [float(x['longitude']), float(x['latitude'])])

grouped

,start_geohash,end_geohash,mean_of_transport,count,start_coords,end_coords
0,u0h43v,u0hhnz,Bicycle,1,"[5.7073974609375, 45.42572021484375]","[5.9271240234375, 45.74432373046875]"
1,u0hhg7,u0hqg7,Car,1,"[5.7733154296875, 45.85418701171875]","[6.1248779296875, 46.20574951171875]"
2,u0hhg7,u0hqge,Car,1,"[5.7733154296875, 45.85418701171875]","[6.1358642578125, 46.20574951171875]"
3,u0hhzk,u0hjqg,Car,1,"[5.9490966796875, 45.85968017578125]","[5.9271240234375, 45.94207763671875]"
4,u0hjzs,u0hnnk,Bicycle,1,"[5.9600830078125, 46.03546142578125]","[5.9051513671875, 46.07940673828125]"
...,...,...,...,...,...,...
7809,u0rcsv,u0rcw0,Car,1,"[11.1126708984375, 46.70013427734375]","[11.1676025390625, 46.67266845703125]"
7810,u0rctf,u0rctg,Walking,1,"[11.1566162109375, 46.68365478515625]","[11.1566162109375, 46.68914794921875]"
7811,u0rhjk,u0rk5n,Train,1,"[10.0799560546875, 47.13409423828125]","[10.3326416015625, 47.14508056640625]"
7812,u0rn1e,u0rn1u,Walking,1,"[9.9151611328125, 47.48016357421875]","[9.9261474609375, 47.48565673828125]"


In [45]:
# Put close geohashes together for 'Train' and 'Car'
grouped = snap_close_geohashes_fast(grouped, mode='Train', max_distance_km=2)
display(grouped)

grouped = snap_close_geohashes_fast(grouped, mode='Car', max_distance_km=2)
display(grouped)

# Merge also bicycle with smaller distance threshold
grouped = snap_close_geohashes_fast(grouped, mode='Bicycle', max_distance_km=0.5)
display(grouped)


,start_geohash,end_geohash,mean_of_transport,count,start_coords,end_coords
0,u0h43v,u0hhnz,Bicycle,1,"[5.7073974609375, 45.42572021484375]","[5.9271240234375, 45.74432373046875]"
1,u0hhg7,u0hqg7,Car,1,"[5.7733154296875, 45.85418701171875]","[6.1248779296875, 46.20574951171875]"
2,u0hhg7,u0hqge,Car,1,"[5.7733154296875, 45.85418701171875]","[6.1358642578125, 46.20574951171875]"
3,u0hhzk,u0hjqg,Car,1,"[5.9490966796875, 45.85968017578125]","[5.9271240234375, 45.94207763671875]"
4,u0hjzs,u0hnnk,Bicycle,1,"[5.9600830078125, 46.03546142578125]","[5.9051513671875, 46.07940673828125]"
...,...,...,...,...,...,...
6707,u0r08x,u0r15w,Train,1,"[9.8712158203125, 46.53533935546875]","[10.0030517578125, 46.61773681640625]"
6708,u0r1tt,u0r604,Train,1,"[10.0909423828125, 46.70013427734375]","[10.2008056640625, 46.77154541015625]"
6709,u0r61n,u0r630,Train,2,"[10.2447509765625, 46.79351806640625]","[10.2447509765625, 46.80450439453125]"
6710,u0rcsu,u0rcsv,Train,1,"[11.1126708984375, 46.69464111328125]","[11.1126708984375, 46.70013427734375]"


,start_geohash,end_geohash,mean_of_transport,count,start_coords,end_coords
0,u0h43v,u0hhnz,Bicycle,1,"[5.7073974609375, 45.42572021484375]","[5.9271240234375, 45.74432373046875]"
1,u0hjzs,u0hnnk,Bicycle,1,"[5.9600830078125, 46.03546142578125]","[5.9051513671875, 46.07940673828125]"
2,u0hjzs,u0hnpd,Bicycle,1,"[5.9600830078125, 46.03546142578125]","[5.9600830078125, 46.06842041015625]"
3,u0hm57,u0hm5k,Walking,3,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90362548828125]"
4,u0hm57,u0hm5m,Walking,2,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90911865234375]"
...,...,...,...,...,...,...
5726,u0rcqp,u0rcr5,Car,1,"[11.1676025390625, 46.66717529296875]","[11.2115478515625, 46.64520263671875]"
5727,u0rcrk,u0rcrz,Car,4,"[11.2225341796875, 46.65069580078125]","[11.2445068359375, 46.66717529296875]"
5728,u0rcr5,u0rct1,Car,1,"[11.2115478515625, 46.64520263671875]","[11.1236572265625, 46.67816162109375]"
5729,u0rcr7,u0rcrh,Car,1,"[11.2225341796875, 46.64520263671875]","[11.2115478515625, 46.65069580078125]"


,start_geohash,end_geohash,mean_of_transport,count,start_coords,end_coords
0,u0hm57,u0hm5k,Walking,3,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90362548828125]"
1,u0hm57,u0hm5m,Walking,2,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90911865234375]"
2,u0hm57,u0hm5s,Walking,1,"[6.1248779296875, 45.89813232421875]","[6.1358642578125, 45.90362548828125]"
3,u0hm5e,u0hm5m,Walking,1,"[6.1358642578125, 45.89813232421875]","[6.1248779296875, 45.90911865234375]"
4,u0hm5k,u0hm5m,Walking,2,"[6.1248779296875, 45.90362548828125]","[6.1248779296875, 45.90911865234375]"
...,...,...,...,...,...,...
5726,u0r08g,u0r08z,Bicycle,1,"[9.8822021484375, 46.51336669921875]","[9.8822021484375, 46.53533935546875]"
5727,u0r08s,u0r08z,Bicycle,1,"[9.8712158203125, 46.51885986328125]","[9.8822021484375, 46.53533935546875]"
5728,u0r08s,u0r090,Bicycle,1,"[9.8712158203125, 46.51885986328125]","[9.8931884765625, 46.49688720703125]"
5729,u0r08x,u0r0c6,Bicycle,1,"[9.8712158203125, 46.53533935546875]","[9.9041748046875, 46.55181884765625]"


In [46]:
# Add random noise to the coordinates for better visualization of the overlapping arcs (for visualizing all the modes of transport together only!)
import numpy as np
def add_noise(coords):
    noise = np.random.normal(0, 0.001, size=2)  # mean 0, std 0.01
    return [coords[0] + noise[0], coords[1] + noise[1]]

grouped['start_coords_noisy'] = grouped['start_coords'].apply(add_noise)
grouped['end_coords_noisy'] = grouped['end_coords'].apply(add_noise)
grouped

,start_geohash,end_geohash,mean_of_transport,count,start_coords,end_coords,start_coords_noisy,end_coords_noisy
0,u0hm57,u0hm5k,Walking,3,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90362548828125]","[6.125475694577723, 45.89824495203909]","[6.126171352802365, 45.904242777586994]"
1,u0hm57,u0hm5m,Walking,2,"[6.1248779296875, 45.89813232421875]","[6.1248779296875, 45.90911865234375]","[6.126127419599018, 45.89922352421547]","[6.124666480930114, 45.908941457103495]"
2,u0hm57,u0hm5s,Walking,1,"[6.1248779296875, 45.89813232421875]","[6.1358642578125, 45.90362548828125]","[6.124055513751359, 45.896151618483366]","[6.136606372076958, 45.904642200714356]"
3,u0hm5e,u0hm5m,Walking,1,"[6.1358642578125, 45.89813232421875]","[6.1248779296875, 45.90911865234375]","[6.136906204744366, 45.898913811562565]","[6.122739278351637, 45.907963712846346]"
4,u0hm5k,u0hm5m,Walking,2,"[6.1248779296875, 45.90362548828125]","[6.1248779296875, 45.90911865234375]","[6.122477007841535, 45.90483112090861]","[6.124110042443824, 45.90970324602076]"
...,...,...,...,...,...,...,...,...
5726,u0r08g,u0r08z,Bicycle,1,"[9.8822021484375, 46.51336669921875]","[9.8822021484375, 46.53533935546875]","[9.882173637769329, 46.5143997253451]","[9.882523892079657, 46.53532769271838]"
5727,u0r08s,u0r08z,Bicycle,1,"[9.8712158203125, 46.51885986328125]","[9.8822021484375, 46.53533935546875]","[9.870808853595149, 46.51842690433944]","[9.881617057051622, 46.53687295357777]"
5728,u0r08s,u0r090,Bicycle,1,"[9.8712158203125, 46.51885986328125]","[9.8931884765625, 46.49688720703125]","[9.872030689103033, 46.51800430223244]","[9.893202649618182, 46.4971727693092]"
5729,u0r08x,u0r0c6,Bicycle,1,"[9.8712158203125, 46.53533935546875]","[9.9041748046875, 46.55181884765625]","[9.870950408483363, 46.535275671815576]","[9.903186587085113, 46.551158782837256]"


In [47]:
# Keep only movements with count > 4
grouped = grouped[grouped['count'] > 4]

In [48]:
# Set color scales for different mean_of_transport (using matplotlib colormaps)
from itertools import count
import matplotlib.pyplot as plt

# color
MODE_COLOR = {
"TRAIN": [180, 255, 180], # green
"TRAM": [255, 220, 170], # orange
"BUS": [255, 160, 160], # red
"WALKING": [150, 200, 255], # blue
"ON_BICYCLE": [170, 255, 255], # cyan
"CAR": [220, 180, 255], # purple
}

# deep color
MODE_DARK_COLOR = {
"TRAIN": [0, 100, 0],
"TRAM": [255, 140, 0],
"BUS": [160, 0, 0],
"WALKING": [0, 80, 180],
"ON_BICYCLE": [0, 150, 150],
"CAR": [90, 0, 160],
}

def mix_color(color1, color2, t):
    return [
        int(color1[0] * (1 - t) + color2[0] * t),
        int(color1[1] * (1 - t) + color2[1] * t),
        int(color1[2] * (1 - t) + color2[2] * t),
    ]

def get_color(mot, count):
    if mot == "Train":
        mode = "TRAIN"
    elif mot == "Tram":
        mode = "TRAM"
    elif mot == "Bus":
        mode = "BUS"
    elif mot == "Walking":
        mode = "WALKING"
    elif mot == "Bicycle":
        mode = "ON_BICYCLE"
    elif mot == "Car":  
        mode = "CAR"
    else:
        mode = "UNKNOWN"

    base = MODE_COLOR.get(mode, [255, 255, 255])
    dark = MODE_DARK_COLOR.get(mode, [0, 0, 0])
    # different times
    if count <= 1:
        t = 0.0 # light
    elif count <= 20:
        t = 0.25 # normal
    elif count <= 50:
        t = 0.5 # deep
    elif count <= 100:
        t = 0.75 # more deep
    else:
        t = 1.0 # deep deep

    return mix_color(base, dark, t)

# Assign color based on mean_of_transport and max count for that transport mode
grouped['color'] = grouped.apply(lambda row: get_color(row['mean_of_transport'], row['count']), axis=1)

/var/folders/hc/bxt7k06n5532f1ws84vbg7180000gn/T/ipykernel_53028/1239346850.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grouped['color'] = grouped.apply(lambda row: get_color(row['mean_of_transport'], row['count']), axis=1)


## Heatmap

In [49]:
# Put all heatmap data to precision 7
df_hm.loc[:, 'geohash'] = df_hm['geohash'].str[:8]

In [50]:
# Get count at each location divided per mode of transport
df_hm_per_mode = df_hm.groupby(["mode_of_transport", "geohash"]).size().reset_index(name="count")

# Translate mode of transport
df_hm_per_mode['mode_of_transport'] = df_hm_per_mode['mode_of_transport'].map(mots)

# Trasform geohash to coordinates
hm_coords = df_hm_per_mode['geohash'].map(geo_to_coords.set_index('geohash')[['latitude', 'longitude']].to_dict(orient='index'))
df_hm_per_mode['coords'] = hm_coords.apply(lambda x: [float(x['longitude']), float(x['latitude'])])

df_hm_per_mode

,mode_of_transport,geohash,count,coords
0,Boat,u0hrzcsw,2,"[6.323490142822266, 46.371660232543945]"
1,Boat,u0hrzcsz,1,"[6.323833465576172, 46.3718318939209]"
2,Boat,u0hrzcv6,1,"[6.324520111083984, 46.37234687805176]"
3,Boat,u0hrzcvu,1,"[6.325206756591797, 46.372690200805664]"
4,Boat,u0hrzcyn,1,"[6.325550079345703, 46.37303352355957]"
...,...,...,...,...
495481,Walking,u0rn1v5t,2,"[9.92563247680664, 47.48934745788574]"
495482,Walking,u0rn4hbn,1,"[9.931812286376953, 47.48814582824707]"
495483,Walking,u0rn4hbw,1,"[9.932498931884766, 47.48814582824707]"
495484,Walking,u0rn4hc1,1,"[9.933185577392578, 47.487287521362305]"


In [51]:
# Get number of elements per each geohash. So we will have two columns: geohash and count
df_hm = df_hm.groupby("geohash", sort=False).size().reset_index(name="count")

# Populate the geohash to coordinates dictionary
geo_to_coords = complete_geo_to_coords(geo_to_coords, df_hm["geohash"].astype(str).unique())

# Transform geohashes to coordinates
coords = df_hm['geohash'].map(geo_to_coords.set_index('geohash')[['latitude', 'longitude']].to_dict(orient='index'))
df_hm['coords'] = coords.apply(lambda x: [float(x['longitude']), float(x['latitude'])])
df_hm

,geohash,count,coords
0,u0m70dcm,9,"[7.406673431396484, 46.94964408874512]"
1,u0m70ddq,1,"[7.408046722412109, 46.948442459106445]"
2,u0m70dfh,3,"[7.407703399658203, 46.949472427368164]"
3,u0m70fqv,11,"[7.425212860107422, 46.94689750671387]"
4,u0m70fqk,36,"[7.424526214599609, 46.946725845336914]"
...,...,...,...
440362,u0k8wqbd,1,"[6.603641510009766, 46.53164863586426]"
440363,u0k8wnf0,1,"[6.594715118408203, 46.53130531311035]"
440364,u0k8wneg,1,"[6.597118377685547, 46.530447006225586]"
440365,u0k8wnsd,1,"[6.598148345947266, 46.53027534484863]"


## Put stuff on the map

In [52]:
# Colormap for heatmap
def set_colormap(cm_name='coolwarm'):
    base = plt.get_cmap(cm_name)
    # Define plasma color map
    plasma_colormap = [base(i)[:3] for i in range(256)]  # Get 256 RGB values
    plasma_colormap = [[int(r*255), int(g*255), int(b*255)] for r, g, b in plasma_colormap]  # Convert to 0-255
    return plasma_colormap

In [53]:
# Display the data using pydeck
layer = pydeck.Layer(
    "ArcLayer",
    data=grouped,
    get_source_position="start_coords_noisy",
    get_target_position="end_coords_noisy",
    get_source_color="color",
    get_target_color="color",
    get_width=1,#"count",
    width_scale=1,#0.1,
    width_min_pixels=2,#0.1,
    width_max_pixels=2,#15,
    pickable=True,
    opacity=0.6,
)

# Show heatmap layer
colormap = set_colormap('inferno')  # Choose desired colormap
heatmap_layer = pydeck.Layer(
    "HeatmapLayer",
    data=df_hm,
    get_position="coords",
    get_weight="count",
    aggregation='SUM',
    radius_pixels=50,
    #intensity=1,
    #threshold=0.03,
    opacity=0.5,
    pickable=False,
    color_range=colormap,  # Apply selected colormap
)

view_state = pydeck.ViewState(
    latitude=46.8182,
    longitude=8.2275,
    zoom=7,
    min_zoom=2,
    max_zoom=17,
    pitch=30,
)

r = pydeck.Deck(
    layers=[
        layer,
        heatmap_layer # Decide if you want to show heatmap on the same visualization as the arcs or separately (probably cleaner, but 2x more different visualizations)
        ], 
    initial_view_state=view_state,
    map_style="https://basemaps.cartocdn.com/gl/positron-gl-style/style.json",
    tooltip={"text": "Mode: {mean_of_transport}\nCount: {count}"}
)

r.to_html('maps/experiment/all_moves.html')
!open -a Arc maps/experiment/all_moves.html

In [54]:
# Now do the same but with train only, and using inferno colormap
modes = grouped['mean_of_transport'].unique()

for mode in modes:
    train_data = grouped[grouped['mean_of_transport'] == mode]

    train_cmap = plt.get_cmap("inferno")
    max_train_count = train_data['count'].max()
    train_data.iloc[:, train_data.columns.get_loc('color')] = train_data['count'].apply(lambda count: [int(c * 255) for c in train_cmap((np.log1p(count) / np.log1p(max_train_count)))[:3]])

    colormap = set_colormap('inferno')
    layer = pydeck.Layer(
        "ArcLayer",
        data=train_data,
        get_source_position="start_coords",
        get_target_position="end_coords",
        get_source_color="color",
        get_target_color="color",
        get_width=1,#"count",
        width_scale=1,#0.1,
        width_min_pixels=3,#0.1,
        width_max_pixels=3,#15,
        pickable=True,
        tooltip={"text": "Mode: {mean_of_transport}\nCount: {count}"},
    )

    # Show heatmap layer
    colormap = set_colormap('inferno')  # Choose desired colormap
    heatmap_layer = pydeck.Layer(
        "HeatmapLayer",
        data=df_hm_per_mode[df_hm_per_mode['mode_of_transport'] == mode],
        get_position="coords",
        get_weight="count",
        aggregation='SUM',
        opacity=0.5,
        pickable=False,
        color_range=colormap,  # Apply selected colormap
    )

    view_state = pydeck.ViewState(
        latitude=46.8182,
        longitude=8.2275,
        zoom=7,
        min_zoom=2,
        max_zoom=17,
        pitch=30,
    )

    r = pydeck.Deck(
        layers=[
            layer,
            heatmap_layer # Decide if you want to show heatmap on the same visualization as the arcs or separately (probably cleaner, but 2x more different visualizations)
            ], 
        initial_view_state=view_state,
        map_style="https://basemaps.cartocdn.com/gl/positron-gl-style/style.json",
    )

    r.to_html(f'maps/experiment/{str.lower(mode)}_moves.html')
    !open -a Arc maps/experiment/{str.lower(mode)}_moves.html